# What drives a readout?

Compare two questions on GPT-2: what supports the answer Paris over London, and what supports a probe trained to distinguish French from British contexts? Then ask what drives one active SAE feature, and check its gradient with forward perturbations.

This CPU example needs `.[examples]`, public GPT-2 weights and the public `gpt2-small-res-jb` SAE. It fits a small illustrative probe, not a validated classifier. Gradient magnitudes depend on the readout's scale; compare their normalized spatial patterns rather than treating their raw norms as comparable evidence.


In [ ]:
MODEL_NAME = "gpt2"
PROMPT = "The Eiffel Tower is in the city of"
PROBE_SITE = "blocks.5.hook_out"
EDIT_SITE = "blocks.3.hook_out"
CAPTURE_SITES = ["blocks.0.hook_out", "blocks.2.hook_out", "blocks.4.hook_out"]
SAE_RELEASE = "gpt2-small-res-jb"
SAE_ID = "blocks.9.hook_resid_pre"

In [ ]:
import matplotlib.pyplot as plt
from IPython import get_ipython
from IPython.display import Markdown, display
import pandas as pd
import torch
from sae_lens import SAE
from sae_lens.analysis.sae_transformer_bridge import SAETransformerBridge

import interpretune as it
import interpretune.analysis
from interpretune.adapters.transformer_lens.backends import TLModelBackend
from interpretune.analysis.backends import Readout
from interpretune.analysis.ops.base import AnalysisBatch

get_ipython().run_line_magic("matplotlib", "inline")
torch.manual_seed(0)
torch.set_num_threads(4)
model = SAETransformerBridge.boot_transformers(MODEL_NAME, device="cpu")
model.enable_compatibility_mode()
model.eval()
backend = TLModelBackend()
tokens = model.to_tokens(PROMPT)
token_labels = [f"{position}: {model.to_string(token)!r}" for position, token in enumerate(tokens[0])]
paris = model.to_single_token(" Paris")
london = model.to_single_token(" London")
print("Prompt:", PROMPT)
print("Model device:", next(model.parameters()).device)
print("Tokens:", token_labels)

## 1. Probe and logit readouts

Fit a unit direction from the difference between mean final-position activations for four French contexts and four British contexts. Keep the Eiffel Tower prompt out of the fit. This direction tests a broad contextual association, not whether the model represents a general concept of France.


In [ ]:
french_contexts = [
    "The capital of France is",
    "The Louvre museum is located in",
    "The French president lives in",
    "The River Seine flows through",
]
british_contexts = [
    "The capital of England is",
    "The British Museum is located in",
    "The British prime minister lives in",
    "The River Thames flows through",
]


def final_state(prompt):
    """Capture the probe site at the final prompt position."""
    with torch.no_grad():
        _, cache = model.run_with_cache(model.to_tokens(prompt), names_filter=[PROBE_SITE])
    return cache[PROBE_SITE][0, -1].detach()


direction = torch.stack([final_state(prompt) for prompt in french_contexts]).mean(0)
direction -= torch.stack([final_state(prompt) for prompt in british_contexts]).mean(0)
direction /= direction.norm()


def answer_difference(logits, activations):
    """Read the final-position Paris versus London logit difference."""
    return (logits[:, -1, paris] - logits[:, -1, london]).sum()


def probe_projection(logits, activations):
    """Read the final-position projection onto the fitted unit direction."""
    return (activations[PROBE_SITE][:, -1] @ direction).sum()


readouts = {
    "Paris minus London": Readout(answer_difference),
    "French-context probe": Readout(probe_projection, sites=(PROBE_SITE,)),
}

The public `it.model_bwd_intervention` op needs a model and a backend. Here a bare PyTorch module supplies those two fields, without a dataset or training runner. The `Readout` is passed in the op's `readout` input. Its scalar uses activations from the same forward pass that the backward pass differentiates.


In [ ]:
module = torch.nn.Module()
module.model = model
module.model_backend = backend


def gradients(readout, interventions=None, sites=None, use_latents=False):
    """Run the public backward op and unpack its named gradient tensors."""
    module.analysis_cfg = it.AnalysisCfg(
        target_op=it.model_bwd_intervention,
        run_inputs={
            "readout": readout,
            "interventions": {} if interventions is None else interventions,
            "grad_capture": CAPTURE_SITES if sites is None else sites,
            "use_latent_models": use_latents,
        },
    )
    result = it.model_bwd_intervention(
        module,
        AnalysisBatch(),
        {"input": tokens},
        0,
    )
    shape_code = result.grad_shape.tolist()
    decoded = {}
    for phase in ("pre", "post"):
        values = result[f"{phase}_intervention_grad_values"]
        shape_offset, value_offset = 1, 0
        decoded[phase] = {}
        for site in result.grad_point_names:
            ndim = shape_code[shape_offset]
            shape = shape_code[shape_offset + 1 : shape_offset + 1 + ndim]
            count = torch.tensor(shape).prod().item()
            decoded[phase][site] = values[value_offset : value_offset + count].reshape(shape)
            value_offset += count
            shape_offset += ndim + 1
    return decoded["pre"], decoded["post"], result.bwd_intervention_metric.item()


baseline = {}
for name, readout in readouts.items():
    baseline[name], _, value = gradients(readout)
    print(f"{name}: scalar {value:.6f}")


def show_map(captured, title):
    """Display a normalized layer-by-position gradient-norm map."""
    matrix = torch.stack([captured[site][0].norm(dim=-1) for site in CAPTURE_SITES])
    normalized = matrix / matrix.sum().clamp_min(1e-12)
    figure, axis = plt.subplots(figsize=(11, 3))
    image = axis.imshow(normalized.numpy(), aspect="auto")
    axis.set_xticks(range(len(token_labels)), token_labels, rotation=55, ha="right")
    axis.set_yticks(range(len(CAPTURE_SITES)), CAPTURE_SITES)
    axis.set_title(title)
    figure.colorbar(image, ax=axis, label="Fraction of summed gradient norms")
    plt.show()
    return normalized


patterns = {name: show_map(captured, name) for name, captured in baseline.items()}
pattern_distance = (patterns["Paris minus London"] - patterns["French-context probe"]).abs().sum().item()
print(f"Spatial pattern L1 difference: {pattern_distance:.6f}")
assert pattern_distance > 1e-4, "The two readouts unexpectedly produced the same normalized pattern"
display(
    Markdown(
        f"**Result:** the normalized maps differ by {pattern_distance:.3f} in L1 distance. "
        "The held-out prompt gives different spatial sensitivities for the probe and the answer logits."
    )
)

The displayed distance measures disagreement on this prompt. A probe can read a contextual association even when later layers transform it before producing the answer logits. Neither gradient map establishes a causal mechanism: it measures local sensitivity to a specified scalar. The feature experiment below supplies a forward control.

## 2. One SAE feature, with a forward check

Attach the pretrained residual SAE at block 9. Compatibility mode matches the SAE's TransformerLens training convention. Select the largest positive final-position feature on this prompt and hold its index fixed for every comparison. Selection is exploratory and gives this feature no semantic label.

The SAE reconstructs its input in every run of this section. Baseline, edited and control runs all use that same reconstruction. The feature is read before decoding, so its upstream derivative tests the encoder path.


In [ ]:
sae = SAE.from_pretrained(SAE_RELEASE, SAE_ID, device="cpu")
module.sae_handles = [sae]
feature_site = model.get_sae_hook_name(sae)
print("SAE model loading convention:", sae.cfg.metadata.model_from_pretrained_kwargs)
with torch.no_grad():
    _, feature_cache = model.run_with_cache_with_saes(tokens, saes=[sae], names_filter=[feature_site, CAPTURE_SITES[1]])
feature_index = int(feature_cache[feature_site][0, -1].argmax())
feature_value = float(feature_cache[feature_site][0, -1, feature_index])
assert feature_value > 0, "No positive SAE feature was available at the final position"


def selected_feature(logits, activations):
    """Read the fixed SAE feature at the final prompt position."""
    return activations[feature_site][:, -1, feature_index].sum()


feature_readout = Readout(selected_feature, sites=(feature_site,))
feature_gradients, _, measured_feature = gradients(feature_readout, use_latents=True)
torch.testing.assert_close(torch.tensor(measured_feature), torch.tensor(feature_value))
feature_pattern = show_map(feature_gradients, f"SAE feature {feature_index}, value {feature_value:.4f}")

At block 2, rank positions by the absolute first-order prediction for shrinking that position's residual activation. Compare a small shrink with a full ablation, reporting both prediction and measurement for every position. The small shrink checks the derivative; a full ablation can leave the local linear regime. Display the highest-ranked position beside the lowest-ranked control, even when the control is not zero.

For a shrink fraction $\epsilon$, the prediction is $-\epsilon \langle g_p, h_p \rangle$. A large gradient norm alone does not predict ablation impact because the intervention direction matters.


In [ ]:
ablation_site = CAPTURE_SITES[1]
clean_activation = feature_cache[ablation_site].detach()
full_predictions = -(feature_gradients[ablation_site] * clean_activation).sum(-1)[0]
ranked_positions = full_predictions.abs().argsort(descending=True).tolist()


def forward_feature(position, fraction):
    """Measure the same feature after shrinking one earlier activation."""

    def shrink(activation, hook):
        changed = activation.clone()
        changed[:, position] *= 1.0 - fraction
        return changed

    with torch.no_grad(), model.saes(saes=[sae]), model.hooks(fwd_hooks=[(ablation_site, shrink)]):
        _, cache = model.run_with_cache(tokens, names_filter=[feature_site])
    return float(cache[feature_site][0, -1, feature_index])


rows = []
for position in ranked_positions:
    for fraction in (0.001, 1.0):
        predicted = fraction * float(full_predictions[position])
        measured = forward_feature(position, fraction) - feature_value
        rows.append(
            {
                "position": position,
                "token": token_labels[position],
                "shrink": fraction,
                "predicted change": predicted,
                "measured change": measured,
                "absolute error": abs(measured - predicted),
            }
        )
comparison = pd.DataFrame(rows)
display(comparison)
print("Top-ranked position:", token_labels[ranked_positions[0]])
print("Lowest-ranked control:", token_labels[ranked_positions[-1]])
small_steps = comparison[comparison["shrink"] == 0.001]
print("Small-step maximum absolute error:", small_steps["absolute error"].max())
assert small_steps["predicted change"].abs().max() > 1e-6
torch.testing.assert_close(
    torch.tensor(small_steps["measured change"].to_numpy()),
    torch.tensor(small_steps["predicted change"].to_numpy()),
    rtol=0.1,
    atol=3e-5,
)
full_steps = comparison[comparison["shrink"] == 1.0]
top_ablation = full_steps.iloc[0]
control_ablation = full_steps.iloc[-1]
display(
    Markdown(
        f"**Result:** 0.1% shrinks have maximum absolute prediction error "
        f"{small_steps['absolute error'].max():.2g}. "
        f"For a full ablation at {top_ablation['token']}, the prediction is "
        f"{top_ablation['predicted change']:.3f} but the measured change is "
        f"{top_ablation['measured change']:.3f}. "
        f"The lowest-ranked control, {control_ablation['token']}, changes the feature by "
        f"{control_ablation['measured change']:.3f}. "
        "The gradient predicts small changes here; its ranking does not establish full-ablation importance."
    )
)

## 3. Edit the gradient and retain the scalar

Reject the probe direction from the gradient at block 3's final position, once under each readout. `reject` removes that component; `project` would keep it. Compare the upstream change with an identity edit. Both edited passes keep the forward scalar unchanged. The result describes which gradient paths pass through the edited site, rather than an intervention that changes the model's answer.


In [ ]:
edit = {
    EDIT_SITE: {
        "intervention_tensor": direction,
        "op": "reject",
        "position_scope": "last_token",
    }
}
identity = {
    EDIT_SITE: {
        "intervention_tensor": torch.ones_like(direction),
        "op": "scale",
        "position_scope": "last_token",
    }
}
edit_rows = []
for name, readout in readouts.items():
    before, after, scalar = gradients(readout, edit)
    clean, unchanged, identity_scalar = gradients(readout, identity)
    torch.testing.assert_close(torch.tensor(scalar), torch.tensor(identity_scalar))
    for site in CAPTURE_SITES:
        torch.testing.assert_close(clean[site], unchanged[site])
        edit_rows.append(
            {
                "readout": name,
                "capture": site,
                "scalar": scalar,
                "gradient change norm": float((after[site] - before[site]).norm()),
            }
        )
display(pd.DataFrame(edit_rows))
for name in readouts:
    rows_for_readout = [row for row in edit_rows if row["readout"] == name]
    assert all(row["gradient change norm"] > 1e-5 for row in rows_for_readout[:2])
    assert rows_for_readout[2]["gradient change norm"] == 0.0
    display(
        Markdown(
            f"**{name}:** rejecting the probe direction changes the gradient norm by "
            f"{rows_for_readout[1]['gradient change norm']:.4f} at block 2, "
            "while the downstream block-4 capture remains unchanged. The identity control is unchanged."
        )
    )

An `add` edit has a second interpretation. Adding a fixed vector $v$ to the gradient at site $s$ gives upstream sites the gradient of $R + \langle v, h_s \rangle$. Check this equivalence explicitly below. A rejection, scale or zero edit instead changes how the chosen derivative propagates through the graph. It does not supply a new, fixed scalar objective valid across inputs.


In [ ]:
for name, readout in readouts.items():
    additive = {
        EDIT_SITE: {
            "intervention_tensor": direction,
            "op": "add",
            "position_scope": "last_token",
        }
    }
    _, added, _ = gradients(readout, additive)

    def augmented(logits, activations, original=readout):
        return original(logits, activations) + (activations[EDIT_SITE][:, -1] @ direction).sum()

    combined = Readout(augmented, sites=tuple(dict.fromkeys((*readout.sites, EDIT_SITE))))
    augmented_gradients, _, _ = gradients(combined)
    upstream_sites = CAPTURE_SITES[:2]
    for site in upstream_sites:
        torch.testing.assert_close(added[site], augmented_gradients[site], rtol=1e-4, atol=1e-5)
    print(f"{name}: additive edit matches the augmented scalar at {upstream_sites}")

## What to take from the measurements

The tables and plots answer three separate questions: where two specified readouts differ in local sensitivity; how well the selected feature's gradient predicts actual forward changes at small and large intervention sizes; and which upstream derivatives change when a gradient path is edited. Keep the prompt, fitted probe, selected feature and intervention direction fixed when comparing runs. Refit and evaluate on held-out prompts before generalizing a finding.

A declarative venngram book needs the consumer's readout compiler in addition to venngram's readout schema. This notebook currently exercises callable readouts; the declared form remains part of the tracked example work until that consumer surface is available.
